In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:90%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# RAG 절차
- https://law.go.kr/법령/소득세법 에서 doc파일로 다운로드 (hwp는 파이썬이 못 읽음, pdf는 한글의 문장,문단,단어 인식이 불가해 짤리는 경우가 많음)
    - 다운로드 후 docx로 변경(새로 저장하기)
   
### [ RAG 구현 절차 ]


```
1.문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt

2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub

```

# 1. 문서를 분리하면서 불러오기(추천)

In [ ]:
import time
start =  time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # 문자 단위로 분리 / 문자 분리 기준 : 문자수 (토큰이 아님) 

loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    # 문서를 분리할 때 1500글자씩 분리
    chunk_overlap=200,  # 200글자정도 겹쳐서 분리하기
    
    )
# 1번째 chunk 1~1,450글자 
# 2번째 chunk 1250~ 2750글자 
documents= loader.load_and_split(text_splitter=text_splitter)
runtime = time.time() - start
print(f"문서 분리 시간 : 총 {runtime}")
# print(document)

# 2. 분리된 문서를 임베딩 → 벡터 데이터베이스 저장
- 임베딩 모델 :  upstage의 openAI API의 text-embedding-3-large (기본:text-embedding-ada-002) 
```
from langchain_upstage import UpstageEmbeddings
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
```
- 백터 데이터 베이스 : chroma

In [2]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large" # 랭체인 사이트에서 추천
#     model="embedding-query"       # 업스테이지 사이트에서 추천
)

In [ ]:
doc_result = embeddings.embed_documents(
    ["점심시간이 곧 다가와요 ", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

In [3]:
%%time
from langchain_chroma import Chroma

# 데이터를 처음 저장할 때 방식
# database = Chroma.from_documents(
#     documents=documents,
#     embedding=embeddings,
#     collection_name='tax-collection',         # 생략시 이름 랜덤 생성
#     persist_directory= './chroma_upstage'     # 생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db 날아감
# )


# 이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function=embeddings,
    collection_name="tax-collection",
    persist_directory='./chroma_upstage'
)

CPU times: total: 547 ms
Wall time: 677 ms


# 3. Vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [4]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [ ]:
retrieved_docs

In [ ]:
retrieved_docs[0].page_content

# 4. 유사도 검색으로 가져온 문서를 질문과 같이 LLM에 전달하여 답변 생성

In [5]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano') # llm 객체 생성

In [6]:
prompt = f"""[idnetity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해 주세요.
[context]는 다음과 같습니다.
{retrieved_docs}
Question:{query}"""

In [7]:
ai_message = llm.invoke(prompt)

In [8]:
print(ai_message.content)

연봉 5천만원인 직장인의 소득세를 계산하려면 기본공제, 근로소득공제, 자녀세액공제 등 여러 공제 항목을 고려해야 합니다. 아래는 주요 절차와 계산 방법입니다.

1. 총급여액: 50,000,000원

2. 근로소득공제 계산
- 최대 2,000만원까지 공제 가능
- 50,000,000원은 2,000만원 초과하므로 공제액은 2,000만원
- 근로소득공제 = 2,000만원

3. 과세표준 산출
- 근로소득공제 후 과세표준 = 50,000,000원 - 20,000,000원 = 30,000,000원

4. 기초공제
- 기본공제(일반 공제액) 약 1,500,000원 (개인별로 다를 수 있으나, 일반적인 값으로 계산)

5. 과세표준에서 인적공제 등을 차감
- 예를 들어 개인공제 150만원 차감 후 과세표준 = 30,000,000원 - 1,500,000원 = 28,500,000원

6. 소득세 계산 (과세표준 기준 세율 적용)
- 1,200만원 이하 구간 세율은 6%
- 1,200만원 초과 4,600만원 이하 구간 세율은 15%
- 5천만원 해당 구간에 대해 누진세율 적용

구체적 세율표 (2023년 기준, 예시):
- 1,200만원까지: 6%
- 1,200만원 초과~4,600만원 이하: 15%
- 4,600만원 초과~8,800만원 이하: 24%
(이하 생략)

계산:
- 6%: 1,200만원 × 6% = 72,000원
- 15%: (4,600만원 - 1,200만원) = 3,400만원 × 15% = 510만원
- 28,500,000원 - 4,600만원 = 1,250만원 (남은 금액)
- 24% 적용: 1,250만원 × 24% = 300만원

세액 합계:
- 72,000원 + 510만원 + 300만원 = 약 810만 원

7. 자녀세액공제 적용
- 자녀 1명인 경우 약 25만원, 2명인 경우 55만원, 3명 이상인 경우 더 높아질 수 있음
- 만약 자녀가 없다면 공제는 없고, 최종 소득세는 약 810만원 수준

최종적으로, 연봉 5천만원인 직장인의 예상 소득세는 약 810만

# 6. Augmentation을 위한 제공되는 Prompt활용하여  langchain으로 답변 생성

In [9]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM 전달(create_retrieval_chain이 대체)
```
Query → retrieval전달(백터 검색 수행) → retrieval 문서 → prompt의 {context}에 삽입 → 전달받은 query → prompt의 {question}에 삽입
```

In [10]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':5}),
    chain_type_kwargs={"prompt":prompt}
)

In [11]:
ai_message = qa_chain.invoke({"query":query})

In [12]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원(50,000,000원) 기준으로, 근로소득공제(최대 2000만원 공제 후 남은 금액) 적용 후 과세표준에 따라 세율이 계산됩니다. 구체적인 소득세액을 계산하려면 공제 및 세율표를 적용해야 하며, 예를 들어 5천만원 연봉 시 세율구간(24%)가 적용될 수 있습니다. 따라서 정확한 세액은 소득공제 후 과세표준에 따라 결정되며, 일반적인 세율 구간과 공제 조건을 고려할 때 대략 수백만 원의 소득세가 부과됩니다.'}